# Sequential Testing & the Peeking Problem

When analysts check experiment results before the planned end-date ("peeking"), conventional p-values lose their meaning. Each extra look inflates the false-positive rate because more chances to cross the significance threshold means more chances to declare a winner that isn't real.

This notebook replays the 78 ASOS experiments at every recorded time snapshot, computes a z-test at each "peek", and compares three decision rules:

| Rule | How it works |
|---|---|
| **Naive** | Declare significance whenever p < 0.05 at any peek |
| **Pocock** | Use a constant but stricter threshold at every peek so the overall alpha stays at 0.05 |
| **O'Brien–Fleming (OBF)** | Use a very strict threshold early and relax it toward 0.05 at the final look |

In [11]:
import math
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set(style='whitegrid', palette='muted')

BASE_DIR = Path('..')
DATA_PATH = BASE_DIR / 'data' / 'asos_digital_experiments_dataset.csv'
OUTPUTS_DIR = BASE_DIR / 'outputs'
PLOTS_DIR = OUTPUTS_DIR / 'plots'

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows · {df['experiment_id'].nunique()} experiments · "
      f"{df['metric_id'].nunique()} metrics · variant IDs: {sorted(df['variant_id'].unique())}")
df.head()

Loaded 24,153 rows · 78 experiments · 4 metrics · variant IDs: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


,experiment_id,variant_id,metric_id,time_since_start,count_c,count_t,mean_c,mean_t,variance_c,variance_t
0,036afc,2,1,1.5,188065.0,186686.0,0.107808,0.107828,0.096186,0.096201
1,036afc,2,1,2.0,245041.0,243694.0,0.131790,0.131435,0.114422,0.114160
2,036afc,2,1,2.5,277237.0,275949.0,0.143065,0.142711,0.122598,0.122345
3,036afc,2,1,3.0,315689.0,314676.0,0.161789,0.160997,0.135613,0.135077
4,036afc,2,1,3.5,338631.0,337715.0,0.172474,0.171067,0.142727,0.141803


## 1 Compute z-score at every peek

Each row already carries cumulative counts, means, and variances for control and treatment at a specific `time_since_start`. We treat every row as a peek and compute the two-sided z-test.

In [12]:
def z_to_p(z):
    """Two-sided p-value from a z-score using the error function."""
    return 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))

df['std_error'] = np.sqrt(
    df['variance_c'] / df['count_c'] + df['variance_t'] / df['count_t']
)
df['lift'] = df['mean_t'] - df['mean_c']
df['z_score'] = df['lift'] / df['std_error']
df['p_value'] = df['z_score'].apply(z_to_p)

pooled_var = (
    ((df['count_c'] - 1) * df['variance_c'] + (df['count_t'] - 1) * df['variance_t'])
    / (df['count_c'] + df['count_t'] - 2)
)
df['std_lift'] = df['lift'] / np.sqrt(pooled_var).replace(0, np.nan)

df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['std_error', 'z_score'])
print(f"{len(df):,} peek-level rows after filtering")
df[['experiment_id', 'variant_id', 'metric_id', 'time_since_start',
    'lift', 'std_lift', 'z_score', 'p_value']].head(10)

23,366 peek-level rows after filtering


,experiment_id,variant_id,metric_id,time_since_start,lift,std_lift,z_score,p_value
0,036afc,2,1,1.5,0.000020,0.000063,0.019390,0.984530
1,036afc,2,1,2.0,-0.000355,-0.001050,-0.366914,0.713683
2,036afc,2,1,2.5,-0.000354,-0.001012,-0.376353,0.706654
3,036afc,2,1,3.0,-0.000792,-0.002152,-0.854217,0.392985
4,036afc,2,1,3.5,-0.001407,-0.003729,-1.533423,0.125172
5,036afc,2,1,4.0,-0.001345,-0.003465,-1.479027,0.139133
6,036afc,2,1,4.5,-0.001672,-0.004273,-1.871714,0.061246
7,036afc,2,1,5.0,-0.001789,-0.004508,-2.035975,0.041753
8,036afc,2,1,5.5,-0.001695,-0.004249,-1.945247,0.051745
9,036afc,2,1,6.0,-0.001903,-0.004705,-2.203523,0.027558


## 2 Number the peeks and assign spending budgets

For each experiment-metric-variant series we number the time snapshots 1 … K and compute two common alpha-spending boundaries:

- **Pocock**: constant threshold at every peek = `alpha / K`
- **O'Brien–Fleming (OBF)**: spend almost nothing early, relax toward alpha at the final look. The z-boundary at look `k` of `K` is `z_{alpha/2} / sqrt(k/K)`, which we convert to a p-value threshold.

In [13]:
ALPHA = 0.05
Z_CRIT = 1.959964  # two-sided 0.05

df = df.sort_values(['experiment_id', 'metric_id', 'variant_id', 'time_since_start'])
df['peek_num'] = df.groupby(['experiment_id', 'metric_id', 'variant_id']).cumcount() + 1
df['total_peeks'] = df.groupby(['experiment_id', 'metric_id', 'variant_id'])['peek_num'].transform('max')
df['info_fraction'] = df['peek_num'] / df['total_peeks']

df['pocock_threshold'] = ALPHA / df['total_peeks']

obf_z_boundary = Z_CRIT / np.sqrt(df['info_fraction'])
df['obf_threshold'] = obf_z_boundary.apply(z_to_p)

df['naive_sig']  = df['p_value'] < ALPHA
df['pocock_sig'] = df['p_value'] < df['pocock_threshold']
df['obf_sig']    = df['p_value'] < df['obf_threshold']

print(f"Median peeks per series: {df.groupby(['experiment_id','metric_id','variant_id'])['peek_num'].max().median():.0f}")
df[['experiment_id', 'metric_id', 'peek_num', 'total_peeks',
    'p_value', 'pocock_threshold', 'obf_threshold',
    'naive_sig', 'pocock_sig', 'obf_sig']].head(10)

Median peeks per series: 63


,experiment_id,metric_id,peek_num,total_peeks,p_value,pocock_threshold,obf_threshold,naive_sig,pocock_sig,obf_sig
0,036afc,1,1,132,0.984530,0.000379,0.000000e+00,False,False,False
1,036afc,1,2,132,0.713683,0.000379,0.000000e+00,False,False,False
2,036afc,1,3,132,0.706654,0.000379,0.000000e+00,False,False,False
3,036afc,1,4,132,0.392985,0.000379,0.000000e+00,False,False,False
4,036afc,1,5,132,0.125172,0.000379,0.000000e+00,False,False,False
5,036afc,1,6,132,0.139133,0.000379,0.000000e+00,False,False,False
6,036afc,1,7,132,0.061246,0.000379,0.000000e+00,False,False,False
7,036afc,1,8,132,0.041753,0.000379,1.776357e-15,True,False,False
8,036afc,1,9,132,0.051745,0.000379,6.084022e-14,False,False,False
9,036afc,1,10,132,0.027558,0.000379,1.072253e-12,True,False,False


## 3 How often does naive peeking trigger a false alarm?

For each experiment-metric-variant series, we check:
- Did the naive rule ever flag significance *before* the final look?
- Did the corrected rules also flag it?

This surfaces the Type-I inflation problem: many series cross p < 0.05 at some early peek, but wouldn't cross the spending boundary.

In [14]:
group_cols = ['experiment_id', 'metric_id', 'variant_id']

early = df[df['peek_num'] < df['total_peeks']].copy()

early_flags = (
    early.groupby(group_cols)
    .agg(
        naive_ever  = ('naive_sig',  'any'),
        pocock_ever = ('pocock_sig', 'any'),
        obf_ever    = ('obf_sig',    'any'),
        first_naive_peek = ('naive_sig', lambda s: s.idxmax() if s.any() else np.nan),
    )
    .reset_index()
)

n_series = len(early_flags)
naive_early  = early_flags['naive_ever'].sum()
pocock_early = early_flags['pocock_ever'].sum()
obf_early    = early_flags['obf_ever'].sum()

print(f"Total experiment-metric-variant series: {n_series}")
print(f"Flagged significant at ≥1 early peek:")
print(f"  Naive  (p < 0.05):          {int(naive_early):>4} ({100*naive_early/n_series:.1f}%)")
print(f"  Pocock (Bonferroni-like):    {int(pocock_early):>4} ({100*pocock_early/n_series:.1f}%)")
print(f"  O'Brien–Fleming:             {int(obf_early):>4} ({100*obf_early/n_series:.1f}%)")

Total experiment-metric-variant series: 381
Flagged significant at ≥1 early peek:
  Naive  (p < 0.05):           222 (58.3%)
  Pocock (Bonferroni-like):      95 (24.9%)
  O'Brien–Fleming:              146 (38.3%)


In [15]:
fig, ax = plt.subplots(figsize=(8, 4))
rules = ['Naive (p<0.05)', 'Pocock', "O'Brien–Fleming"]
counts = [int(naive_early), int(pocock_early), int(obf_early)]
colors = ['#e45756', '#f58518', '#4c78a8']
bars = ax.barh(rules, counts, color=colors, edgecolor='white')
ax.set_xlabel('Series flagged significant at an early peek')
ax.set_title('Early stopping signals: naive vs. corrected rules')
for bar, c in zip(bars, counts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(c), va='center', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'early_stopping_comparison.png', dpi=150)
plt.show()

/var/folders/xh/yvr1g2bd7d10yql092bn6pjc0000gn/T/ipykernel_34707/3276063243.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3b Were the early alarms right? Confusion matrix vs. final-look truth

An early alarm is only useful if the experiment is *actually* significant at the end. We use the final-peek p-value as ground truth and classify each early alarm as a **true positive** (correctly early-called) or **false positive** (would have been called not significant at the end).

In [ ]:
final_look = (
    df[df['peek_num'] == df['total_peeks']]
    [group_cols + ['p_value']]
    .rename(columns={'p_value': 'final_p'})
)
final_look['final_sig'] = final_look['final_p'] < ALPHA

alarm_accuracy = early_flags.merge(final_look, on=group_cols)

def classify(row, rule_col):
    flagged = row[rule_col]
    true_sig = row['final_sig']
    if flagged and true_sig:
        return 'True Positive'
    elif flagged and not true_sig:
        return 'False Positive'
    elif not flagged and true_sig:
        return 'Missed (False Neg)'
    else:
        return 'True Negative'

for rule, col in [('naive', 'naive_ever'), ('pocock', 'pocock_ever'), ('obf', 'obf_ever')]:
    alarm_accuracy[f'{rule}_outcome'] = alarm_accuracy.apply(lambda r, c=col: classify(r, c), axis=1)

print("Early-alarm accuracy vs. final-look ground truth\n")
for rule_name, col in [('Naive', 'naive_outcome'), ('Pocock', 'pocock_outcome'), ("O'Brien-Fleming", 'obf_outcome')]:
    counts = alarm_accuracy[col].value_counts()
    total = len(alarm_accuracy)
    print(f"  {rule_name}:")
    for label in ['True Positive', 'False Positive', 'Missed (False Neg)', 'True Negative']:
        n = counts.get(label, 0)
        print(f"    {label:<20s} {n:>4} ({100*n/total:.1f}%)")
    tp = counts.get('True Positive', 0)
    fp = counts.get('False Positive', 0)
    fn = counts.get('Missed (False Neg)', 0)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"    {'Precision':<20s} {precision:.1%}")
    print(f"    {'Recall':<20s} {recall:.1%}")
    print()

In [ ]:
outcome_order = ['True Positive', 'False Positive', 'Missed (False Neg)', 'True Negative']
outcome_colors = {'True Positive': '#4c78a8', 'False Positive': '#e45756',
                  'Missed (False Neg)': '#f58518', 'True Negative': '#72b7b2'}

rows = []
for rule_name, col in [('Naive', 'naive_outcome'), ('Pocock', 'pocock_outcome'), ("OBF", 'obf_outcome')]:
    counts = alarm_accuracy[col].value_counts()
    for label in outcome_order:
        rows.append({'Rule': rule_name, 'Outcome': label, 'Count': counts.get(label, 0)})
outcome_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(outcome_order))
width = 0.25
for i, rule in enumerate(['Naive', 'Pocock', 'OBF']):
    vals = outcome_df[outcome_df['Rule'] == rule]['Count'].values
    bars = ax.bar(x + i * width, vals, width, label=rule,
                  color=[outcome_colors[o] for o in outcome_order], alpha=0.5 + 0.2 * i,
                  edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(v), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(outcome_order, fontsize=10)
ax.set_ylabel('Number of series')
ax.set_title('Early-alarm accuracy: were the peeking signals right?')
ax.legend(title='Rule')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'alarm_accuracy_confusion.png', dpi=150)
plt.show()

In [ ]:
per_exp = (
    alarm_accuracy.groupby('experiment_id')
    .agg(
        series          = ('final_sig', 'count'),
        final_sig       = ('final_sig', 'sum'),
        naive_alarms    = ('naive_ever', 'sum'),
        naive_TP        = ('naive_outcome', lambda s: (s == 'True Positive').sum()),
        naive_FP        = ('naive_outcome', lambda s: (s == 'False Positive').sum()),
        pocock_alarms   = ('pocock_ever', 'sum'),
        pocock_TP       = ('pocock_outcome', lambda s: (s == 'True Positive').sum()),
        pocock_FP       = ('pocock_outcome', lambda s: (s == 'False Positive').sum()),
        obf_alarms      = ('obf_ever', 'sum'),
        obf_TP          = ('obf_outcome', lambda s: (s == 'True Positive').sum()),
        obf_FP          = ('obf_outcome', lambda s: (s == 'False Positive').sum()),
    )
    .reset_index()
    .sort_values('naive_FP', ascending=False)
)
per_exp = per_exp.astype({c: int for c in per_exp.columns if c != 'experiment_id'})

print("Per-experiment alarm accuracy (sorted by naive false positives)\n")
display(per_exp)
print(f"\nExperiments where naive peeking produced ≥1 false positive: "
      f"{(per_exp['naive_FP'] > 0).sum()} / {len(per_exp)}")

## 4 Standardized lift stability over time

A key insight: early peeks tend to produce **inflated** effect sizes. We plot average absolute standardized lift by information fraction to show how estimates settle down as the experiment matures.

In [16]:
df['info_bin'] = pd.cut(df['info_fraction'], bins=20, labels=False)
stability = (
    df.groupby('info_bin')
    .agg(
        mean_abs_std_lift = ('std_lift', lambda s: s.abs().mean()),
        mean_abs_z        = ('z_score', lambda s: s.abs().mean()),
        pct_naive_sig     = ('naive_sig', 'mean'),
    )
    .reset_index()
)
stability['info_fraction'] = (stability['info_bin'] + 0.5) / 20

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=False)

axes[0].plot(stability['info_fraction'], stability['mean_abs_std_lift'],
             marker='o', color='#4c78a8')
axes[0].set_title('Avg |standardized lift|')
axes[0].set_xlabel('Information fraction (0 = start, 1 = end)')
axes[0].set_ylabel('|Std lift|')

axes[1].plot(stability['info_fraction'], stability['mean_abs_z'],
             marker='o', color='#f58518')
axes[1].set_title('Avg |z-score|')
axes[1].set_xlabel('Information fraction')
axes[1].set_ylabel('|z|')

axes[2].bar(stability['info_fraction'], stability['pct_naive_sig'] * 100,
            width=0.04, color='#e45756', edgecolor='white')
axes[2].axhline(5, ls='--', color='gray', lw=1, label='nominal 5%')
axes[2].set_title('% naive significant')
axes[2].set_xlabel('Information fraction')
axes[2].set_ylabel('% p < 0.05')
axes[2].legend()

plt.suptitle('How estimates stabilize as experiments mature', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'lift_stability_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/xh/yvr1g2bd7d10yql092bn6pjc0000gn/T/ipykernel_34707/2445696436.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5 Deep dive: boundary plots for selected experiments

For a handful of experiments we plot the running z-score against the Pocock and OBF boundaries. This is the classic "monitoring chart" that sequential-testing teams use in practice.

In [17]:
from scipy.stats import norm

top_experiments = (
    df.groupby('experiment_id')['total_peeks'].first()
    .nlargest(4).index.tolist()
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, exp_id in enumerate(top_experiments):
    ax = axes[i]
    sub = df[(df['experiment_id'] == exp_id) & (df['metric_id'] == 1)].copy()
    if sub.empty:
        sub = df[df['experiment_id'] == exp_id].copy()
    sub = sub.sort_values('time_since_start')
    K = sub['total_peeks'].iloc[0]
    info_frac = sub['info_fraction']

    pocock_z = norm.ppf(1 - (ALPHA / K) / 2)
    obf_z = Z_CRIT / np.sqrt(info_frac)

    ax.plot(info_frac, sub['z_score'].abs(), color='#4c78a8', lw=1.5, label='|z-score|')
    ax.axhline(Z_CRIT, ls='--', color='gray', lw=1, label=f'Naive z={Z_CRIT:.2f}')
    ax.axhline(pocock_z, ls='-.', color='#f58518', lw=1.2, label=f'Pocock z={pocock_z:.2f}')
    ax.plot(info_frac, obf_z, ls=':', color='#e45756', lw=1.5, label='OBF boundary')

    ax.set_title(f'Experiment {exp_id} (K={K})', fontsize=11)
    ax.set_xlabel('Information fraction')
    ax.set_ylabel('|z|')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(bottom=0)

plt.suptitle('Sequential monitoring charts: z-score vs. spending boundaries', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'sequential_monitoring_charts.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/xh/yvr1g2bd7d10yql092bn6pjc0000gn/T/ipykernel_34707/3404782703.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 Lift overestimation from early stopping

If we stopped at the first peek where naive p < 0.05, what lift would we have reported vs. the final lift? This quantifies how much early stopping inflates effect sizes.

In [18]:
first_sig = (
    df[df['naive_sig']]
    .sort_values('time_since_start')
    .groupby(group_cols, as_index=False)
    .first()
    .rename(columns={'std_lift': 'early_std_lift', 'lift': 'early_lift',
                     'peek_num': 'early_peek', 'info_fraction': 'early_info_frac'})
    [group_cols + ['early_std_lift', 'early_lift', 'early_peek', 'early_info_frac']]
)

final = (
    df[df['peek_num'] == df['total_peeks']]
    .rename(columns={'std_lift': 'final_std_lift', 'lift': 'final_lift'})
    [group_cols + ['final_std_lift', 'final_lift', 'total_peeks']]
)

comparison = first_sig.merge(final, on=group_cols)
comparison['lift_inflation'] = comparison['early_std_lift'].abs() - comparison['final_std_lift'].abs()

avg_inflation = comparison['lift_inflation'].mean()
median_early_frac = comparison['early_info_frac'].median()

print(f"Series that were ever naive-significant: {len(comparison)}")
print(f"Median information fraction at first naive flag: {median_early_frac:.2f}")
print(f"Average |std lift| inflation from early stopping: {avg_inflation:+.4f} SD")

Series that were ever naive-significant: 225
Median information fraction at first naive flag: 0.10
Average |std lift| inflation from early stopping: +0.0044 SD


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(comparison['lift_inflation'], bins=30, color='#e45756', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='gray', lw=1)
axes[0].axvline(avg_inflation, color='black', ls='--', lw=1.2,
                label=f'mean = {avg_inflation:+.4f}')
axes[0].set_title('Lift inflation: |early std lift| − |final std lift|')
axes[0].set_xlabel('Inflation (SD units)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(comparison['early_info_frac'], bins=20, color='#4c78a8', edgecolor='white', alpha=0.85)
axes[1].axvline(median_early_frac, color='black', ls='--', lw=1.2,
                label=f'median = {median_early_frac:.2f}')
axes[1].set_title('When does the first naive flag appear?')
axes[1].set_xlabel('Information fraction at first p < 0.05')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'lift_inflation_early_stopping.png', dpi=150)
plt.show()

/var/folders/xh/yvr1g2bd7d10yql092bn6pjc0000gn/T/ipykernel_34707/2622193809.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 Summary statistics

Aggregate the key numbers for the resume bullet.

In [20]:
print("="*60)
print("SEQUENTIAL TESTING SUMMARY")
print("="*60)
print(f"Experiments analyzed:            {df['experiment_id'].nunique()}")
print(f"Experiment-metric-variant series: {n_series}")
print(f"Total peek-level rows:           {len(df):,}")
print()
print("Early-peek false alarms (before final look):")
print(f"  Naive  (p < 0.05):             {int(naive_early)} / {n_series} ({100*naive_early/n_series:.1f}%)")
print(f"  Pocock correction:             {int(pocock_early)} / {n_series} ({100*pocock_early/n_series:.1f}%)")
print(f"  O'Brien–Fleming correction:    {int(obf_early)} / {n_series} ({100*obf_early/n_series:.1f}%)")
print()
print(f"Avg |std lift| inflation from premature stopping: {avg_inflation:+.4f} SD")
print(f"Median info fraction at first naive flag:         {median_early_frac:.0%}")
print("="*60)

SEQUENTIAL TESTING SUMMARY
Experiments analyzed:            78
Experiment-metric-variant series: 381
Total peek-level rows:           23,366

Early-peek false alarms (before final look):
  Naive  (p < 0.05):             222 / 381 (58.3%)
  Pocock correction:             95 / 381 (24.9%)
  O'Brien–Fleming correction:    146 / 381 (38.3%)

Avg |std lift| inflation from premature stopping: +0.0044 SD
Median info fraction at first naive flag:         10%


## Takeaways

1. **Naive peeking inflates false positives.** Many experiment series cross p < 0.05 at some early look, even when the final result is not significant.
2. **Alpha-spending boundaries control the damage.** Pocock and O'Brien–Fleming dramatically reduce early false flags while still allowing genuinely strong effects to be caught early.
3. **Early stopping overestimates effect sizes.** The standardized lift at the first naive flag is on average inflated compared to the final estimate—stakeholders relying on premature results would over-promise impact.
4. **Wait for ~85%+ information fraction.** Most estimates stabilize past that point; stopping earlier requires a corrected boundary.